In [30]:
import keras.ops
import numpy as np


a = np.array([[ 3.0225076e+02, -2.8821728e-01,  0.0000000e+00],
       [ 8.7628273e+01,  4.8883435e+02,  1.0000000e+00],
       [ 5.1000000e+02,  5.0000000e+02,  1.0000000e+00]])

a[:, 0] = np.where(a[:, 2] < 1, 0, a[:, 0])
a[:, 1] = np.where(a[:, 2] < 1, 0, a[:, 1])
a[:, :2]

array([[  0.      ,   0.      ],
       [ 87.628273, 488.83435 ],
       [510.      , 500.      ]])

In [323]:
G = 1.0
advantage = []
for _ in range(5):
    G *= 0.99
    advantage.append(G)
advantage = advantage[::-1]
advantage

[0.9509900498999999, 0.96059601, 0.9702989999999999, 0.9801, 0.99]

In [318]:
import kymnasium as kym
from tensorflow.keras import layers, ops, models
from tensorflow_probability import distributions as tfd
import numpy as np
import tensorflow as tf

class Agent(kym.Agent):
    def __init__(
            self,
            turn: int,
            model: str = None,
            training: bool = False,
            gamma: float = 0.99
    ):
        self.turn = turn
        self.training = training
        self.gamma = gamma
        self.actor = models.load_model(model) if model else self._build_actor()
        self.buf_state, self.buf_action, self.buf_action_prob = [], [], []

    def save(self, path: str):
        pass

    def train(self, is_win: bool):
        G = 1.0 if is_win else -1.0
        advantage = []
        for _ in range(len(self.buf_state)):
            G *= self.gamma
            advantage.append(G)
        advantage = advantage[::-1]

        buf_state = keras.ops.convert_to_tensor(self.buf_state)
        buf_action = keras.ops.convert_to_tensor(self.buf_action)
        buf_action_prob = keras.ops.convert_to_tensor(self.buf_action_prob)

    def _train_actor(self):
        with tf.GradientTape() as tape:
            ratio = keras.ops.exp(

            )

    @classmethod
    def load(cls, path: str) -> 'Agent':
        pass

    @classmethod
    def _preprocess(cls, stones: np.ndarray):
        stones = np.array(stones)

        for i in range(1):
            stones[:, i] = np.where(stones[:, 2] < 1, 0, stones[:, i])
        stones = stones / 300.0 - 1.0
        stones = np.ravel(stones[:, :2])
        return stones

    @classmethod
    def _postprocess(cls, angle, power):
        angle = ops.squeeze(angle)
        power = ops.squeeze(power)
        angle *= 180.0
        power *= 2500.0
        angle = np.clip(angle, -180.0, 180.0)
        power = np.clip(power, 1.0, 2500.0)
        return angle, power

    @classmethod
    def _build_actor(cls):
        input_turn = layers.Input(shape=(1,))
        input_black = layers.Input(shape=(6,))
        input_white = layers.Input(shape=(6,))
        concat = layers.Concatenate()([input_turn, input_black, input_white])
        layer = layers.Dense(units=512, kernel_initializer='he_normal', activation="relu")(concat)
        layer = layers.Dense(units=256, kernel_initializer='he_normal', activation="relu")(layer)
        layer = layers.Dense(units=128, kernel_initializer='he_normal', activation="relu")(layer)
        output_index = layers.Dense(units=3, activation='softmax')(layer)
        output_angle_mean = layers.Dense(units=1, activation='tanh')(layer)
        output_angle_log_sigma = layers.Dense(units=1, activation='tanh')(layer)
        output_power_mean = layers.Dense(units=1, activation='sigmoid')(layer)
        output_power_log_sigma = layers.Dense(units=1, activation='tanh')(layer)

        return models.Model(
            inputs=[input_turn, input_black, input_white],
            outputs=[output_index, output_angle_mean, output_angle_log_sigma, output_power_mean, output_power_log_sigma]
        )

    def act(self, observation, info):
        turn, black, white = observation['turn'], observation['black'], observation['white']
        if self.turn != turn:
            return None

        black, white = self._preprocess(black), self._preprocess(white)
        turn, black, white = ops.expand_dims(turn, axis=0), ops.expand_dims(black, axis=0), ops.expand_dims(white, axis=0)
        idx_probs, angle_mean, angle_log_sigma, power_mean, power_log_sigma = self.actor([turn, black, white])

        if self.training:
            idx = np.random.choice(3, p=np.squeeze(idx_probs))
            angle = angle_mean + ops.exp(angle_log_sigma) * np.random.normal()
            power = power_mean + ops.exp(power_log_sigma) * np.random.normal()
        else:
            idx = np.random.choice(np.squeeze(idx_probs) == np.max(idx_probs))
            angle = angle_mean
            power = power_mean

        action_probs = ops.log(
            np.squeeze(idx_probs)[idx]
        ) + tfd.Normal(
            loc=angle_mean, scale=ops.exp(angle_log_sigma)
        ).log_prob(angle) + tfd.Normal(
            loc=power_mean, scale=ops.exp(power_log_sigma)
        ).log_prob(power)

        self.buf_state.append((turn, black, white))
        self.buf_action.append((idx, angle, power))
        self.buf_action_prob.append(action_probs)

        angle, power = self._postprocess(angle, power)

        return {
            'turn': self.turn,
            'angle': angle,
            'power': power,
            'index': int(idx)
        }





In [321]:
import gymnasium as gym


env = gym.make('kymnasium/AlKkaGi-3x3-v0', render_mode='rgb_array', obs_type='custom')
obs, info = env.reset()
agent = Agent(turn=0, training=True)
agent.act(obs, info)
#obs

tf.Tensor([[-5.8740573]], shape=(1, 1), dtype=float32)


{'turn': 0,
 'angle': np.float32(-180.0),
 'power': np.float32(2500.0),
 'index': 0}